# MNLP Base Model

In [ ]:
import sys
from pathlib import Path
import torch

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "preprocessing" / "align.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the project checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import *
from src.model import *

In [ ]:
### CONFIG PARAMS

# Set source language 'S", pivot language 'P', and target language 'T'
# Names based on the languages in the Ted database
S = "az"
P = "uk"
T = "be"

DIRECTION_S_P = f"{S}-{P}"
DIRECTION_P_T = f"{P}-{T}"
DIRECTION_S_T = f"{S}-{T}"


# Number of training epochs for source to pivot language and pivot to target language
NR_EPOCHS_S_P = 10
NR_EPOCHS_P_T = 10

# Sizes of vocabulary for source, pivot and target language
VOCAB_SIZE_S = 4000
VOCAB_SIZE_P = 4000
VOCAB_SIZE_T = 4000

MAX_LENGTH_S_P = 20
MAX_LENGTH_P_T = 20


# Automatically sets training on GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# Create train set for the first and second model
train_S_P = get_language_data_aligned('train', S, P)[:100]
train_P_T = get_language_data_aligned("train", P, T)[:100]

# Get test data data to and from pivot language, as well as the source to target language chain.
test_S_P = get_language_data_aligned("test", S, P)
test_P_T = get_language_data_aligned("test", P, T)
test_S_T = get_language_data_aligned("test", S, T)


In [ ]:
# Create vocabulary from languages' train datasets
# Both to and from pivot language data is included.
S_vocab = Vocabulary([s[0] for s in train_S_P])
P_vocab = Vocabulary([s[1] for s in train_S_P] + [s[0] for s in train_P_T])
T_vocab = Vocabulary([s[1] for s in train_P_T])

In [ ]:
# Create model from source to pivot language
model_S_P = Seq2SeqModel(
    src_vocab=S_vocab,
    tgt_vocab=P_vocab
)

# Create model from pivot to target language
model_P_T = Seq2SeqModel(
    src_vocab=P_vocab,
    tgt_vocab=T_vocab
)

In [ ]:
# # Fit models with training pairs
model_S_P.fit(pairs=train_S_P,epochs=NR_EPOCHS_S_P, device=device)
model_P_T.fit(pairs=train_P_T,epochs=NR_EPOCHS_P_T, device=device)

In [ ]:
from src.evaluation import evaluate_direction

test_pred_S_P = translate_corpus(model_S_P, [s for s, _ in test_S_P], max_length=MAX_LENGTH_S_P, device=device)
test_pred_P_T = translate_corpus(model_P_T, [s for s, _ in test_P_T], max_length=MAX_LENGTH_P_T, device=device)
test_pred_pivot_S_T = translate_corpus(model_S_P, [s for s, _ in test_S_T], max_length=MAX_LENGTH_S_P, device=device)

convert = [((s[0]), (s[1])) for s in test_pred_pivot_S_T]
test_pred_target_S_T = translate_corpus(model_P_T, [s for s, _ in convert], max_length=MAX_LENGTH_S_P, device=device)

out_dir = ROOT / "results" / "translations" / "pivot"

In [ ]:
# Saves translations of direct S->P, P->T, and the pivot translation S->T
save_translations(out_dir, DIRECTION_S_P, [s for s, _ in test_S_P], [t for _, t in test_S_P], test_pred_S_P)
save_translations(out_dir, DIRECTION_P_T, [s for s, _ in test_P_T], [t for _, t in test_P_T], test_pred_P_T)
save_translations(out_dir, DIRECTION_S_T, [s for s, _ in test_S_T], [t for _, t in test_S_T], test_pred_target_S_T)



In [ ]:
from comet import download_model, load_from_checkpoint

from sacrebleu.metrics import BLEU, CHRF

In [ ]:
from datetime import datetime

# Helper function based on the pipeline code. 
# Calculate the three metrics, then returns a dict containing it,
# as well as saving the output to a file.

def get_scores(folder, direction, name="", save_to_file=True):
  scores = {}
  if name == "":
    name = direction

  gpu_available = 1 if torch.cuda.is_available() else 0
  time = datetime.now().strftime('%d-%m_%H:%M')
    
  name = ("scores_{}_{}".format(name, time))

  
  for metric in ("bleu", "chrf", "comet"):
    
    try:
      metric_output = evaluate_direction(out_dir, direction, metric, gpus=gpu_available)
    except Exception as e:
      metric_output = str(e)

    scores[metric] = metric_output

  if save_to_file:
    with open(str(out_dir) + "/" + name + ".txt", "a") as score_file:
      for metric, output in scores.items():
        print(metric + ":", file=score_file)
        print(output, file=score_file)

    print("Saved scores to: " + name)

  return scores



In [ ]:
scores_S_P = get_scores(out_dir, DIRECTION_S_P, name=DIRECTION_S_P)
scores_P_T = get_scores(out_dir, DIRECTION_P_T, name=DIRECTION_P_T)
scores_S_T = get_scores(out_dir, DIRECTION_S_T, name=DIRECTION_S_T)
